# RAG Pipeline Demo

This notebook demonstrates the local PDF RAG flow used by this project:

1. Extract PDF text into chunks.
2. Generate local embeddings and a SQLite vector database.
3. Embed a user question.
4. Retrieve the most relevant chunks.
5. Send retrieved context to OpenAI and return an answer with citations.

Before running the final LLM cell, make sure `backend/.env` contains your `OPENAI_API_KEY`.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
BACKEND_DIR = PROJECT_ROOT / "backend"
EXTRACTED_DIR = BACKEND_DIR / "extracted_text"

print(PROJECT_ROOT)
print(BACKEND_DIR)

## 1. Extract PDFs Into Text Chunks

In [ ]:
from backend.read_text import main as extract_text

extract_text()

chunks_path = EXTRACTED_DIR / "rag_chunks.jsonl"
print(f"Chunks file exists: {chunks_path.exists()}")
print(f"Chunk count: {sum(1 for _ in chunks_path.open(encoding='utf-8'))}")

## 2. Generate Embeddings And Save Local Vector Database

This project uses an offline open-source TF-IDF + SVD embedding pipeline and stores vectors in SQLite.

In [ ]:
from backend.generate_embeddings import main as generate_embeddings

generate_embeddings()

vector_db_path = EXTRACTED_DIR / "rag_vector_store.sqlite"
embedding_model_path = EXTRACTED_DIR / "rag_embedding_model.joblib"

print(f"Vector DB exists: {vector_db_path.exists()}")
print(f"Embedding model exists: {embedding_model_path.exists()}")

## 3. Embed A User Question

In [ ]:
from backend.embed_question import embed_question

question = "When must a new employee complete Form I-9?"
question_embedding = embed_question(question)

print(f"Question: {question}")
print(f"Embedding dimensions: {len(question_embedding)}")
print(question_embedding[:8])

## 4. Retrieve Relevant Chunks

In [ ]:
from backend.embed_question import retrieve_chunks

chunks = retrieve_chunks(question, top_k=3)

for chunk in chunks:
    print("-" * 80)
    print(f"Score: {chunk['score']}")
    print(f"Source: {chunk['source']}, page {chunk['page']}")
    print(chunk['text'][:700])

## 5. Ask The LLM With Citations

This cell calls OpenAI using `backend/.env`. The returned `answer` should include inline citations, and the structured `citations` field can be used by an app UI.

In [ ]:
from backend.ask_llm import answer_question

result = answer_question(question, top_k=3, include_chunks=False)

print(result["answer"])
print("\nCitations:")
for citation in result["citations"]:
    print(f"- {citation['label']}")

## Optional: Dry Run Without Calling OpenAI

In [ ]:
dry_run = answer_question(question, top_k=3, dry_run=True)

print(dry_run.keys())
print(dry_run["citations"])
print(dry_run["messages"][0])